# Attack Characterization: AgentPoison, MINJA, InjecMEM

This notebook evaluates and characterizes the three attack methodologies:

- **AgentPoison** (Chen et al., NeurIPS 2024, arXiv:2407.12784): Trigger-based embedding-space poisoning
- **MINJA** (Dong et al., NeurIPS 2025, arXiv:2503.03704): Query-only injection with bridging steps
- **InjecMEM** (ICLR 2026, openreview:QVX6hcJ2um): Single-interaction retriever-agnostic anchor

Evaluation uses a FAISS-backed vector memory system with sentence-transformer (all-MiniLM-L6-v2) embeddings to compute paper-faithful ASR-R, ASR-A, and ASR-T metrics.

**ASR Metric Definitions** (Chen et al., NeurIPS 2024):
- **ASR-R**: Fraction of victim queries that retrieve ≥1 adversarial entry in top-k
- **ASR-A**: Fraction of retrievals that execute the adversarial action (given retrieval)
- **ASR-T**: End-to-end hijacking rate = successful_hijacks / total_queries

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# add src/ to path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from evaluation.retrieval_sim import RetrievalSimulator, generate_agentpoison_passage, generate_minja_passage, generate_injecmem_passage
from data.synthetic_corpus import SyntheticCorpus, VICTIM_QUERIES, BENIGN_QUERIES
from memory_systems.vector_store import VectorMemorySystem

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

SEED = 42
CORPUS_SIZE = 200
TOP_K = 5
N_POISON_BASE = 5

print(f'setup complete: corpus_size={CORPUS_SIZE}, top_k={TOP_K}, n_poison_base={N_POISON_BASE}')

## 1. Build Corpus and Run Evaluation

In [ ]:
sim = RetrievalSimulator(
    corpus_size=CORPUS_SIZE,
    top_k=TOP_K,
    n_poison_per_attack=N_POISON_BASE,
    seed=SEED,
)

print(f'corpus stats: {sim.get_corpus_stats()}')

# run evaluation for all three attacks
attack_results = {}
attack_types = ['agent_poison', 'minja', 'injecmem']

for at in attack_types:
    m = sim.evaluate_attack(at)
    attack_results[at] = m
    n_entries = len(sim._generate_poison_entries(at, sim._victim_queries))
    poison_rate = n_entries / (CORPUS_SIZE + n_entries) * 100
    print(f'\n{at} ({n_entries} entries, {poison_rate:.1f}% poison rate):')
    print(f'  ASR-R = {m.asr_r:.3f}')
    print(f'  ASR-A = {m.asr_a:.3f}')
    print(f'  ASR-T = {m.asr_t:.3f}')
    print(f'  Benign Accuracy = {m.benign_accuracy:.3f}')

## 2. Per-Attack ASR Comparison (Bar Chart)

In [ ]:
attack_labels = {'agent_poison': 'AgentPoison\n(Chen et al., 2024)', 'minja': 'MINJA\n(Dong et al., 2025)', 'injecmem': 'InjecMEM\n(ICLR 2026)'}
metrics_keys = ['asr_r', 'asr_a', 'asr_t']
metric_labels = ['ASR-R (Retrieval)', 'ASR-A (Action | Retrieval)', 'ASR-T (End-to-End)']
colors = ['#E74C3C', '#F39C12', '#27AE60']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Attack Success Rate Comparison Across Methodologies', fontsize=14, fontweight='bold')

x = np.arange(len(attack_types))
width = 0.5

for ax, mkey, mlabel, color in zip(axes, metrics_keys, metric_labels, colors):
    values = [getattr(attack_results[at], mkey) for at in attack_types]
    bars = ax.bar(x, values, width, color=color, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Attack Type')
    ax.set_ylabel(mlabel)
    ax.set_title(mlabel)
    ax.set_xticks(x)
    ax.set_xticklabels([attack_labels[at] for at in attack_types], fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../visualization/fig_attack_asr_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_attack_asr_comparison.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_attack_asr_comparison')

## 3. Per-Query Retrieval Success Heatmap

In [ ]:
# build detailed per-query retrieval results
corpus = SyntheticCorpus(seed=SEED)
entries = corpus.generate_benign_entries(CORPUS_SIZE)
victim_queries = corpus.get_victim_queries()

per_query_results = {}  # attack_type -> list[bool]

for at in attack_types:
    poison_entries = sim._generate_poison_entries(at, sim._victim_queries)
    mem = VectorMemorySystem()
    mem.add_batch(entries)
    for pe in poison_entries:
        mem.store(pe['key'], pe['content'])
    poison_keys = [pe['key'] for pe in poison_entries]
    
    retrieved = []
    for q in victim_queries:
        result = mem.poison_retrieval_test(q['query'], poison_keys, top_k=TOP_K)
        retrieved.append(1 if result['retrieved_any_poison'] else 0)
    per_query_results[at] = retrieved

# create heatmap
heatmap_data = np.array([per_query_results[at] for at in attack_types])
query_labels = [f"{q['query'][:35]}..." if len(q['query']) > 35 else q['query'] for q in victim_queries]

fig, ax = plt.subplots(figsize=(18, 4))
im = ax.imshow(heatmap_data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(victim_queries)))
ax.set_xticklabels(query_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(attack_types)))
ax.set_yticklabels([attack_labels[at].replace('\n', ' ') for at in attack_types])
ax.set_title('Per-Query Poison Retrieval Success (Green=Retrieved, Red=Not Retrieved)', fontsize=12)
plt.colorbar(im, ax=ax, label='Retrieved (1) / Not Retrieved (0)')
plt.tight_layout()
plt.savefig('../visualization/fig_per_query_retrieval_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_per_query_retrieval_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_per_query_retrieval_heatmap')

## 4. Cosine Similarity Distribution: Poison vs. Benign Entries

In [ ]:
# compute cosine similarity distributions for each attack's poison content vs victim queries
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Cosine Similarity Distributions: Poison vs Benign Entries for Victim Queries', fontsize=12)

for ax, at in zip(axes, attack_types):
    poison_entries = sim._generate_poison_entries(at, sim._victim_queries)
    mem = VectorMemorySystem()
    mem.add_batch(entries)
    for pe in poison_entries:
        mem.store(pe['key'], pe['content'])
    poison_keys = set(pe['key'] for pe in poison_entries)
    
    # collect top-1 scores for all queries
    poison_scores = []
    benign_top1_scores = []
    
    for q in victim_queries:
        results = mem.search(q['query'], top_k=TOP_K)
        for r in results:
            if r['key'] in poison_keys:
                poison_scores.append(r['score'])
        # top benign score
        benign_results = [r for r in results if r['key'] not in poison_keys]
        if benign_results:
            benign_top1_scores.append(benign_results[0]['score'])
    
    ax.hist(benign_top1_scores, bins=15, alpha=0.6, color='steelblue', label='Benign top-1', density=True)
    if poison_scores:
        ax.hist(poison_scores, bins=10, alpha=0.6, color='crimson', label='Poison entries', density=True)
    ax.set_xlabel('Cosine Similarity')
    ax.set_ylabel('Density')
    ax.set_title(attack_labels[at].replace('\n', '\n'))
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../visualization/fig_cosine_similarity_distributions.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_cosine_similarity_distributions.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_cosine_similarity_distributions')

## 5. Attack Stealthiness: ASR-R vs Benign Accuracy Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

attack_colors = {'agent_poison': '#E74C3C', 'minja': '#3498DB', 'injecmem': '#27AE60'}
attack_markers = {'agent_poison': 'o', 'minja': 's', 'injecmem': '^'}

for at in attack_types:
    m = attack_results[at]
    ax.scatter(
        m.asr_r,
        m.benign_accuracy,
        c=attack_colors[at],
        marker=attack_markers[at],
        s=200,
        zorder=5,
        label=attack_labels[at].replace('\n', ' '),
        edgecolors='black',
        linewidths=1,
    )
    ax.annotate(
        f'  ASR-T={m.asr_t:.2f}',
        (m.asr_r, m.benign_accuracy),
        fontsize=9,
    )

# ideal region: high ASR-R + high benign accuracy (top-right)
ax.fill_between([0.5, 1.0], [0.8, 0.8], [1.0, 1.0], alpha=0.08, color='green', label='Ideal attack zone')

ax.set_xlabel('ASR-R (Retrieval Success Rate)', fontsize=12)
ax.set_ylabel('Benign Accuracy (Stealthiness)', fontsize=12)
ax.set_title('Attack Effectiveness vs. Stealthiness Trade-off', fontsize=13)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0.5, 1.05)
ax.legend(loc='lower left', fontsize=9)
ax.axhline(y=0.9, color='gray', linestyle=':', alpha=0.5, linewidth=0.8)
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, linewidth=0.8)
ax.set_aspect('auto')

plt.tight_layout()
plt.savefig('../visualization/fig_attack_stealthiness_tradeoff.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_attack_stealthiness_tradeoff.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_attack_stealthiness_tradeoff')

## 6. Poison Rate Ablation: ASR-R vs Number of Poison Entries

In [ ]:
# evaluate ASR-R across different poison entry counts (ablation)
poison_base_values = [1, 2, 3, 5, 7, 10]
ablation_results = {at: {'asr_r': [], 'asr_t': [], 'poison_count': [], 'poison_rate': []} for at in attack_types}

for n_base in poison_base_values:
    sim_ab = RetrievalSimulator(corpus_size=CORPUS_SIZE, top_k=TOP_K, n_poison_per_attack=n_base, seed=SEED)
    for at in attack_types:
        entries_ab = sim_ab._generate_poison_entries(at, sim_ab._victim_queries)
        m_ab = sim_ab.evaluate_attack(at)
        n_p = len(entries_ab)
        ablation_results[at]['asr_r'].append(m_ab.asr_r)
        ablation_results[at]['asr_t'].append(m_ab.asr_t)
        ablation_results[at]['poison_count'].append(n_p)
        ablation_results[at]['poison_rate'].append(n_p / (CORPUS_SIZE + n_p))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ablation Study: Attack Effectiveness vs. Poison Entry Count', fontsize=13)

for ax, metric, ylabel in zip(axes, ['asr_r', 'asr_t'], ['ASR-R (Retrieval)', 'ASR-T (End-to-End)']):
    for at in attack_types:
        ax.plot(
            ablation_results[at]['poison_count'],
            ablation_results[at][metric],
            marker=attack_markers[at],
            color=attack_colors[at],
            label=attack_labels[at].replace('\n', ' '),
            linewidth=2,
            markersize=8,
        )
    ax.set_xlabel('Number of Adversarial Entries in Memory')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../visualization/fig_poison_count_ablation.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_poison_count_ablation.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_poison_count_ablation')

## 7. Save Experiment Results

In [ ]:
import json
from dataclasses import asdict

results_to_save = {
    'experiment': 'attack_characterization',
    'corpus_size': CORPUS_SIZE,
    'top_k': TOP_K,
    'n_poison_base': N_POISON_BASE,
    'seed': SEED,
    'attack_metrics': {at: asdict(attack_results[at]) for at in attack_types},
    'ablation_results': ablation_results,
}

output_path = '../visualization/attack_characterization_results.json'
with open(output_path, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f'results saved to {output_path}')
print('\n=== Summary ===')
for at in attack_types:
    m = attack_results[at]
    print(f'{at:15s}: ASR-R={m.asr_r:.3f}  ASR-A={m.asr_a:.3f}  ASR-T={m.asr_t:.3f}  Benign={m.benign_accuracy:.3f}')